In [ ]:
import pandas as pd

import re

In [ ]:
df = pd.read_parquet('../data/raw/text-to-cypher/neo4j-2024v1/train-00000-of-00001.parquet')
df

In [ ]:
# count and shows unique database_reference_alias column values
df['database_reference_alias'].value_counts(dropna=False)

In [ ]:
# print how many rows have database_reference_alias
print(f"Rows with database_reference_alias: {df['database_reference_alias'].notnull().sum()}")

In [ ]:
# filter rows that starts with "Node properties:"
mask = df['schema'].str.startswith("Node properties:")
to_remove = df[~mask]
df = df[mask]
print(f"Removing {len(to_remove)} rows that do not start with 'Node properties:'")
print(f"Remaining rows: {len(df)}")
print("Removed rows:")
to_remove

In [ ]:
# replace each ` character with nothing
df['schema'] = df['schema'].str.replace('`', '')

In [ ]:
# schema_regex =r'Node properties:\n(?P<nodes>(?:(?:.*)\n)+)Relationship properties:\n(?P<rel_properties>(?:(?:.*)\n)+)The relationships:\n(?P<relationships>(?:(?:.*)\n?)+)'
# node_regex = r'- \*\*(?P<node>[a-zA-Z\_]+)\*\*\n(?P<properties>(?:  - (?:.+)\n)+)'
# properties_regex = r'  - `(?P<property>[a-zA-Z\_]+)`: (?P<property_type>[a-zA-Z\_]+) (?P<example>\s*Example: \".+\")?(?:Min: (?P<min>.+), Max: (?P<max>.+))?\n?'

In [ ]:
removed_groups_regex = re.compile(r'\?P<\w+>')

In [ ]:
# property_regex = r'  - `(?P<property_name>.+)`: (?P<property_type>\w+) (?P<example>\s*Example: \".*\")?(?P<extra>.+)?\n?'
property_regex = r'  - (?P<property_name>\w+): (?P<property_type>\w+) (?P<extra>.+)?\n?'
# replace all named groups in node_regex with ?: to make them non-capturing

# temp_single_prop_regex = removed_groups_regex.sub('?:', property_regex)
entity_regex = rf'- \*\*(?P<entity>\w+)\*\*\n+(?P<properties>(?:{property_regex})+)'
cleaned_entity_regex = removed_groups_regex.sub('?:', entity_regex)
print(cleaned_entity_regex, end="\n\n")

# temp_node_regex = removed_groups_regex.sub('?:', entity_regex)
# schema_regex =rf'Node properties:\n(?P<nodes>(?:{entity_regex}\n+)+)Relationship properties:\n(?P<rel_properties>(?:(?:.*)\n)+)The relationships:\n(?P<relationships>(?:(?:.*)\n?)+)'

schema_regex =rf'Node properties:\n+(?P<nodes>(?:{cleaned_entity_regex}\n+)+)Relationship properties:\n+(?P<rel_properties>(?:{cleaned_entity_regex}\n+)*)The relationships:\n+(?P<relationships>(?:(?:.*)\n?)+)'

# temp_schema_regex = removed_groups_regex.sub('?:', schema_regex)

print(schema_regex)

# Schema regex

754 da rimuovere

In [ ]:
# remove rows that do not match with the context regex
to_keep = df['schema'].apply(lambda x: re.match(schema_regex, x) is not None)
to_remove = df[~to_keep]
print(f"Removing {len(to_remove)} rows that do not match the schema regex")
to_remove

In [ ]:
entity_regex2 = r'\w+ (?P<properties_dict>\{(?:(?:\w+\:\s+\w+),?\s*)+\})'
cleaned_entity_regex2 = removed_groups_regex.sub('?:', entity_regex2)
print(cleaned_entity_regex2, end="\n\n")
schema2_regex =rf'Node properties:\n+(?P<nodes>(?:{cleaned_entity_regex2}\n+)+)Relationship properties:\n+(?P<rel_properties>(?:{cleaned_entity_regex2}\n+)*)The relationships:\n+(?P<relationships>(?:(?:.*)\n?)+)'
print(schema2_regex)

In [ ]:
to_keep2 = to_remove['schema'].apply(lambda x: re.match(schema2_regex, x) is not None)
to_remove2 = to_remove[~to_keep2]
print(f"Removing {len(to_remove2)} rows that do not match the schema2 regex")
to_remove2

In [ ]:
df = df[to_keep]
df